<a href="https://colab.research.google.com/github/irembagcivan/Veri-Bilimi-Bootcamp/blob/main/Python_Oturum_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#RNN

##Tokenizer

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
veri = [
    'Bu bir örnek cümle.',
    'Başka bir cümle de var.',
    'Farklı bir örnek metin.'
]

In [ ]:
tokenizer = Tokenizer() # oov_token="<OOV>"
tokenizer.fit_on_texts(veri)
sayisal_karsilik = tokenizer.texts_to_sequences(veri)

In [ ]:
print("Kelime indeksleri : ",tokenizer.word_index)

In [ ]:
for sayi in sayisal_karsilik:
  print(sayi)

In [ ]:
sabit_uzunluk = pad_sequences(sayisal_karsilik,maxlen=5)

In [ ]:
for sayi in sabit_uzunluk:
  print(sayi)

##Embedding

In [ ]:
from keras.layers import Embedding
import numpy as np

In [ ]:
giris_boyutu = len(tokenizer.word_index) + 1

boyut = 300

embedding_layer = Embedding(
    input_dim=giris_boyutu,
    output_dim=boyut,
    input_length=5
)

In [ ]:
kimlikli_veri = embedding_layer(np.array(sabit_uzunluk))

In [ ]:
print("Embedding Katman Sonucu : ")
print(kimlikli_veri)

##RNN

In [ ]:
!pip install joblib
!pip install colorama

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from keras.layers import SimpleRNN, Dense
from keras.utils import to_categorical
import joblib
from colorama import Fore, Back, Style
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

In [ ]:
veri_yolu = '/content/drive/MyDrive/Tech İstanbul/Phishing_Email.csv'
model_yolu = '/content/drive/MyDrive/Tech İstanbul/rnn.h5'

In [ ]:
mail_verisi = pd.read_csv(veri_yolu)
mail_verisi = mail_verisi.where((pd.notnull(mail_verisi)),'')

In [ ]:
mail_verisi.loc[mail_verisi['Email Type']=='Safe Email','Email Type'] = 0
mail_verisi.loc[mail_verisi['Email Type']=='Phishing Email','Email Type'] = 1

In [ ]:
X = mail_verisi['Email Text']
y = mail_verisi['Email Type']

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X)
indeksli_veri = tokenizer.texts_to_sequences(X)

In [ ]:
word_index = tokenizer.word_index
kelime_uzunlugu = 1000
sabit_uzunluk = pad_sequences(indeksli_veri, maxlen=kelime_uzunlugu)

In [ ]:
y = to_categorical(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(sabit_uzunluk,y, test_size=0.2, random_state=42)

In [ ]:
model = Sequential()
model.add(Embedding(len(word_index)+1, output_dim=128, input_length=kelime_uzunlugu))
model.add(SimpleRNN(64,activation='relu'))
model.add(Dense(2,activation='softmax'))
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [ ]:
model.fit(X_train,y_train, epochs=5, batch_size=32, validation_data=(X_test,y_test))

In [ ]:
model.save(model_yolu)

In [ ]:
import tensorflow as tf
model = tf.keras.models.load_model(model_yolu, compile=False)
#Eğitim yapılacağı zaman compile parametresi kaldır ve aşağıdakini çalıştır.
#model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [ ]:
while True:
  giris = input('Email mesajını giriniz : ')
  sayisal_kelime = tokenizer.texts_to_sequences([giris])
  sabit_uzunluk = pad_sequences(sayisal_kelime, maxlen=kelime_uzunlugu)
  tahmin = model.predict(sabit_uzunluk)
  if np.argmax(tahmin) == 1:
    print(Fore.RED, 'Bu bir kimlik avı emaili olabilir.')
  else:
    print(Fore.GREEN, 'Bu bir güvenli emaildir.')
  print(Fore.BLUE, 'Doğruluk puanı : ', tahmin[0][np.argmax(tahmin)])
